# Demo 01 | Tool Use | Agentic Workflows.

01 - TOOL USE: o modelo escolhe a ferramenta, o schema decide se ela roda.

O QUE ESTA DEMO MOSTRA, EM 4 ETAPAS:
  1. Um CATALOGO de duas ferramentas descritas por schema (o que o MCP, na
     Aula 2, vai transformar em protocolo de rede).
  2. O modelo recebe uma pergunta em portugues e ESCOLHE qual ferramenta
     chamar e com quais argumentos - via function calling nativo.
  3. O payload proposto pelo modelo passa por VALIDACAO (pydantic) antes de
     tocar o banco. Essa fronteira e o guardrail.
  4. Um payload deliberadamente invalido e BARRADO, para o aluno ver a
     fronteira funcionando.

Dados reais: dados/curso_financeiro.db, tabela clientes (German Credit, 1000
solicitantes de credito).

> A chave da OpenRouter usada abaixo é a **chave temporária da turma**, embutida de propósito para a aula funcionar sem setup. Ela é descartável: não é uma credencial pessoal, e é revogada depois do curso.


## Ambiente (instalar pacotes)

In [37]:
!pip install -q openai python-dotenv pydantic matplotlib pandas

## Configuração

In [38]:
import json
import os
import sqlite3
import sys
import time
from pathlib import Path

import pandas as pd
import matplotlib
matplotlib.use("Agg")  # backend sem janela: salva o grafico em arquivo
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError

# Opções para exibir todas as linhas e colunas sem truncamento
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)

load_dotenv()

CHAVE_TEMPORARIA = "REMOVIDA"   # chave da aula; o ambiente sempre vence
MODELO = os.getenv("OPENROUTER_MODEL", "openai/gpt-5.6-luna")
CHAVE  = os.getenv("OPENROUTER_API_KEY") or CHAVE_TEMPORARIA
PASTA  = Path(globals().get("__file__", ".")).resolve().parent
DB     = next(p for p in (PASTA / "dados" / "curso_financeiro.db",
                          *(a / "dados" / "curso_financeiro.db" for a in PASTA.parents),
                          PASTA / "curso_financeiro.db") if p.exists())
FIG    = PASTA / "01-tool-use-posicao.png"

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=CHAVE)


def chamar_llm(mensagens, ferramentas=None, tentativas=2):
    """Unico helper do script: chama o modelo e nunca deixa vazar traceback.

    Em sala, uma oscilacao de wifi nao pode derrubar a aula: aqui ela vira
    uma mensagem clara e o script segue (ou encerra com dignidade).
    """
    for tentativa in range(1, tentativas + 1):
        try:
            resp = client.chat.completions.create(
                model=MODELO, messages=mensagens, tools=ferramentas,
            )
            return resp.choices[0].message
        except Exception as erro:
            print(f"[AVISO] Falha na chamada ao modelo ({tentativa}/{tentativas}): "
                  f"{type(erro).__name__}: {erro}")
            if tentativa < tentativas:
                time.sleep(2)
    sys.exit("[ERRO] Nao foi possivel falar com o OpenRouter. Verifique a rede e a chave.")


print(f"TOOL USE - modelo: {MODELO}")

TOOL USE - modelo: openai/gpt-5.6-luna


## Olhada nos dados

In [39]:
# Re-executar a consulta para obter os dados
con = sqlite3.connect(DB)
con.row_factory = sqlite3.Row
cursor = con.cursor()

transacoes_raw = con.execute("SELECT * FROM transacoes").fetchall()

# Query para obter os nomes das tabelas
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tabelas = cursor.fetchall()

print("Tabelas no banco de dados:")
for tabela in tabelas:
    print(f"- {tabela[0]}")

# Converter os objetos sqlite3.Row em dicionários e depois em DataFrame
transacoes_dict = [dict(row) for row in transacoes_raw]
df_transacoes = pd.DataFrame(transacoes_dict)

# Exibir o DataFrame
display(df_transacoes.head(5))
display(df_transacoes.tail(5))

Tabelas no banco de dados:
- clientes
- transacoes
- eventos_acesso
- reclamacoes


,transacao_id,cliente_id,data,tipo,valor,cidade,canal,conta_destino
0,1,256,2025-06-11,saque,800.00,Sao Paulo,atm,NaN
1,2,256,2025-06-11,saque,650.00,Rio de Janeiro,atm,NaN
2,3,256,2025-06-11,saque,900.00,Belo Horizonte,atm,NaN
3,4,37,2025-06-21,pix_enviado,5656.36,Belo Horizonte,app,conta_mula_9
4,5,512,2025-06-20,pix_enviado,3576.87,Sao Paulo,app,conta_mula_9


,transacao_id,cliente_id,data,tipo,valor,cidade,canal,conta_destino
216,217,581,2025-07-20,deposito,385.93,Recife,app,NaN
217,218,581,2025-07-25,deposito,2460.39,Rio de Janeiro,atm,NaN
218,219,581,2025-06-08,pix_enviado,1659.42,Sao Paulo,web,NaN
219,220,898,2025-07-21,deposito,1975.31,Recife,atm,NaN
220,221,898,2025-06-12,compra,678.11,Rio de Janeiro,agencia,NaN


## ETAPA 1 - O CATALOGO DE FERRAMENTAS

Cada ferramenta e um contrato: nome + descricao + schema de argumentos.
O modelo NAO ve o banco; ve apenas esta descricao.

In [40]:
class ConsultarClienteArgs(BaseModel):
    """Argumentos de consultar_cliente."""
    cliente_id: int = Field(description="Identificador inteiro do cliente")


class ListarTransacoesArgs(BaseModel):
    """Argumentos de listar_transacoes."""
    cliente_id: int = Field(description="Identificador inteiro do cliente")


def consultar_cliente(cliente_id: int) -> dict:
    """Implementacao real da ferramenta: le o cadastro no SQLite."""
    con = sqlite3.connect(DB)
    con.row_factory = sqlite3.Row
    linha = con.execute(
        "SELECT cliente_id, idade, proposito, valor_credito, duracao_meses, "
        "conta_corrente, historico_credito, risco FROM clientes WHERE cliente_id = ?",
        (cliente_id,),
    ).fetchone()
    con.close()
    return dict(linha) if linha else {}


def listar_transacoes(cliente_id: int) -> list:
    """Implementacao real da ferramenta: le as transacoes no SQLite."""
    con = sqlite3.connect(DB)
    con.row_factory = sqlite3.Row
    linhas = con.execute(
        "SELECT data, tipo, valor, cidade, canal FROM transacoes "
        "WHERE cliente_id = ? ORDER BY data",
        (cliente_id,),
    ).fetchall()
    con.close()
    return [dict(x) for x in linhas]


# O catalogo no formato que o modelo entende. Repare: e o MESMO conteudo dos
# schemas pydantic acima, so que serializado como JSON Schema.
CATALOGO = [
    {"type": "function", "function": {
        "name": "consultar_cliente",
        "description": "Retorna o cadastro de credito de um cliente (idade, proposito, "
                       "valor, historico, risco).",
        "parameters": ConsultarClienteArgs.model_json_schema()}},
    {"type": "function", "function": {
        "name": "listar_transacoes",
        "description": "Lista as transacoes de um cliente (data, tipo, valor, cidade, canal).",
        "parameters": ListarTransacoesArgs.model_json_schema()}},
]
IMPLEMENTACOES = {"consultar_cliente": (ConsultarClienteArgs, consultar_cliente),
                  "listar_transacoes": (ListarTransacoesArgs, listar_transacoes)}

print("\n[1] CATALOGO OFERECIDO AO MODELO")
for f in CATALOGO:
    print(f"    - {f['function']['name']}: {f['function']['description']}...")


[1] CATALOGO OFERECIDO AO MODELO
    - consultar_cliente: Retorna o cadastro de credito de um cliente (idade, proposito, valor, historico, risco)....
    - listar_transacoes: Lista as transacoes de um cliente (data, tipo, valor, cidade, canal)....


## ETAPA 2 - O MODELO ESCOLHE

Nao dizemos qual ferramenta usar. A pergunta e ambigua de proposito: fala em
"movimentacoes", nao em "transacoes".

In [41]:
PERGUNTA = ("O gerente quer entender o perfil de credito do cliente 96 antes de "
            "aprovar a proposta de emprestimo. Comece pelo cadastro dele.")

print(f"\n[2] PERGUNTA EM LINGUAGEM NATURAL\n    \"{PERGUNTA}\"")

mensagem = chamar_llm(
    [{"role": "system", "content": "Voce e um assistente de analise de credito. "
                                   "Use as ferramentas disponiveis para responder."},
     {"role": "user", "content": PERGUNTA}],
    ferramentas=CATALOGO,
)

if not mensagem.tool_calls:
    print("    O modelo respondeu sem chamar ferramenta:")
    print(f"    {mensagem.content}")
    sys.exit(0)

escolha = mensagem.tool_calls[0]
nome = escolha.function.name
argumentos_crus = escolha.function.arguments
print(f"    -> ferramenta escolhida: {nome}")
print(f"    -> argumentos propostos: {argumentos_crus}")


[2] PERGUNTA EM LINGUAGEM NATURAL
    "O gerente quer entender o perfil de credito do cliente 96 antes de aprovar a proposta de emprestimo. Comece pelo cadastro dele."
    -> ferramenta escolhida: consultar_cliente
    -> argumentos propostos: {"cliente_id":96}


## ETAPA 3 - A FRONTEIRA DE CONFIANCA

O que veio do modelo e uma SUGESTAO, nao um comando. Validamos o schema
antes de deixar qualquer coisa chegar ao banco.

In [42]:
print("\n[3] VALIDACAO ANTES DE EXECUTAR")

if nome not in IMPLEMENTACOES:
    sys.exit(f"    RECUSADO: o modelo pediu '{nome}', que nao esta no catalogo.")

Schema, funcao = IMPLEMENTACOES[nome]
try:
    args = Schema(**json.loads(argumentos_crus))
    print(f"    validacao OK -> executando {nome}({args.cliente_id})")
    resultado = funcao(args.cliente_id)
except (ValidationError, json.JSONDecodeError) as erro:
    sys.exit(f"    RECUSADO pelo schema, ferramenta NAO executada:\n{erro}")

if not resultado:
    sys.exit(f"    A ferramenta rodou, mas o cliente {args.cliente_id} nao existe no banco.")

print("    resultado da ferramenta (dado real do banco):")
for chave, valor in resultado.items():
    print(f"        {chave:20s} {valor}")


[3] VALIDACAO ANTES DE EXECUTAR
    validacao OK -> executando consultar_cliente(96)
    resultado da ferramenta (dado real do banco):
        cliente_id           96
        idade                58
        proposito            negocio
        valor_credito        15945
        duracao_meses        54
        conta_corrente       baixa
        historico_credito    sem_creditos_ou_quitados
        risco                ruim


## ETAPA 4 - O GUARDRAIL BARRANDO UM PAYLOAD RUIM

Simulamos um modelo que alucinou o argumento. O schema recusa.

In [46]:
print("\n[4] TESTE DE PAYLOAD INVALIDO (modelo 'alucinando' o argumento)")

# TRES falhas de natureza diferente, sempre na mesma ordem:
#   1. erro de TIPO      - string onde se espera inteiro;
#   2. erro de CONTRATO  - o campo obrigatorio nem veio;
#   3. erro de DOMINIO   - e numero, mas nao e inteiro.
PAYLOADS_INVALIDOS = [
    ('{"cliente_id": "noventa e seis"}', {"cliente_id": "noventa e seis"}),
    ('{"conta": 96}',                    {"conta": 96}),
    ('{"cliente_id": 96.5}',             {"cliente_id": 96.5}),
]

for rotulo, payload_ruim in PAYLOADS_INVALIDOS:
    try:
        ConsultarClienteArgs(**payload_ruim)
        print(f"    {rotulo:34s} -> PASSOU (nao deveria!)")
    except ValidationError as erro:
        motivo = erro.errors()[0]
        print(f"    {rotulo:34s} -> BARRADO ({motivo['type']})")


[4] TESTE DE PAYLOAD INVALIDO (modelo 'alucinando' o argumento)
    {"cliente_id": "noventa e seis"}   -> BARRADO (int_parsing)
    {"conta": 96}                      -> BARRADO (missing)
    {"cliente_id": 96.5}               -> BARRADO (int_from_float)


## GRAFICO: onde este cliente cai na carteira de 1000 solicitantes.

In [44]:
con = sqlite3.connect(DB)
valores = [linha[0] for linha in con.execute("SELECT valor_credito FROM clientes")]
con.close()

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.hist(valores, bins=40, color="#2c5282", alpha=0.85)
ax.axvline(resultado["valor_credito"], color="#c0392b", linewidth=2,
           label=f"cliente {resultado['cliente_id']} = {resultado['valor_credito']}")
ax.set_title("Posicao do cliente na carteira (German Credit, n=1000)")
ax.set_xlabel("valor_credito")
ax.set_ylabel("clientes")
ax.legend()
fig.tight_layout()
# fig.show()
fig.savefig(FIG, dpi=110)
plt.close(fig)
print(f"\n[grafico salvo] {FIG.name}")

print("LICAO: o modelo decide O QUE chamar; o schema decide SE aquilo roda.")
print("Na Aula 2, esse mesmo catalogo vira um servidor MCP acessivel pela rede.")


[grafico salvo] 01-tool-use-posicao.png
LICAO: o modelo decide O QUE chamar; o schema decide SE aquilo roda.
Na Aula 2, esse mesmo catalogo vira um servidor MCP acessivel pela rede.
